# Multi-Embodiment Experiment: H1 to G1 Stand Task

**⚠️ IMPORTANT LIMITATION:** This notebook demonstrates the architectural challenge of cross-embodiment transfer.

Due to dimension mismatch between H1 (19 joints) and G1 (37 joints), direct model transfer is **not currently possible** without modifications to the architecture. See the Analysis Notes section for details and potential solutions.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

os.environ["TORCHDYNAMO_INLINE_INBUILT_NN_MODULES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MUJOCO_GL"] = "egl"


import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tensordict import TensorDict, from_module

torch.autograd.set_detect_anomaly(True)
torch.set_float32_matmul_precision("high")

from fast_td3.fast_td3_utils import (
    EmpiricalNormalization,
)

from fast_td3 import Critic
from fast_td3.actors import ActorEGNN, Actor

## Configuration

**Important:** Due to EGNN architecture constraints, `source_robot` and `target_robot` must currently be the same.

Use this notebook to evaluate a trained model on the same robot type:
- H1 model → H1 environment, OR
- G1 model → G1 environment

In [ ]:
from fast_td3.hyperparams import HumanoidBenchArgs# NOTE: source_robot and target_robot must be the same due to dimension mismatch# Change both to "g1" if evaluating a G1 modelsource_robot = "h1"source_task = "stand-v0"# Target robot (evaluate model on this)target_robot = "h1"  # Must match source_robot for nowtarget_task = "stand-v0"# Create args for the TARGET environment (g1-stand-v0)args = HumanoidBenchArgs(    env_name=f"{target_robot}-{target_task}",    total_timesteps=50000,    render_interval=5000,    eval_interval=1000,    num_envs=16,    batch_size=8192,    actor_hidden_dim=384,)print(f"Evaluating model trained on {source_robot}-{source_task} on environment {target_robot}-{target_task}")

In [ ]:
amp_enabled = args.amp and args.cuda and torch.cuda.is_available()
amp_device_type = (
    "cuda"
    if args.cuda and torch.cuda.is_available()
    else "mps" if args.cuda and torch.backends.mps.is_available() else "cpu"
)
amp_dtype = torch.bfloat16 if args.amp_dtype == "bf16" else torch.float16

scaler = GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

if not args.cuda:
    device = torch.device("cpu")
else:
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{args.device_rank}")
    elif torch.backends.mps.is_available():
        device = torch.device(f"mps:{args.device_rank}")
    else:
        raise ValueError("No GPU available")
print(f"Using device: {device}")

## Environment Setup

Create the G1 stand environment for evaluation.

In [ ]:
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

env_type = "humanoid_bench"
eval_envs = HumanoidBenchEnv(args.env_name, args.num_envs, device=device)
render_env = HumanoidBenchEnv(args.env_name, 1, render_mode="rgb_array", device=device)

n_act = eval_envs.num_actions
n_obs = eval_envs.num_obs if type(eval_envs.num_obs) == int else eval_envs.num_obs[0]
if eval_envs.asymmetric_obs:
    n_critic_obs = (
        eval_envs.num_privileged_obs
        if type(eval_envs.num_privileged_obs) == int
        else eval_envs.num_privileged_obs[0]
    )
else:
    n_critic_obs = n_obs
action_low, action_high = -1.0, 1.0

print(f"Target environment: {args.env_name}")
print(f"Observation space: {n_obs}")
print(f"Action space: {n_act}")

## Model Loading

**⚠️ LIMITATION:** Due to dimension mismatch, you cannot load an H1 model and use it on G1 directly.

This cell shows how to load a model for the **same robot** as the training robot.
To use this notebook:
- Train on H1 → Evaluate on H1, OR
- Train on G1 → Evaluate on G1

For cross-embodiment transfer, architectural changes are needed (see Analysis Notes).

**Note:** Update the checkpoint_path and ensure source_robot matches target_robot.

In [ ]:
checkpoint_path = "./models/egnn_h1-run-v0_16envs_300001steps_0f9e38_295000.pt"

# If you don't have a trained model yet, you can create a placeholder:
# For demonstration purposes, we'll create the actor architecture
# but you'll need an actual trained checkpoint to get meaningful results

obs_normalizer = EmpiricalNormalization(shape=n_obs, device=device)
xanchor_normalizer = EmpiricalNormalization(shape=(20, 3), device=device)

# Actor setup - using SOURCE robot (h1) architecture
# but it will receive observations from TARGET robot (g1)
actor = ActorEGNN(
    num_envs=args.num_envs,
    batch_size=args.batch_size,
    device=device,
    hidden_dim=64,
    n_layers=4,
    act_fn="relu",
    robot=source_robot,  # Use source robot (h1) for the graph structure
    env_name=f"{source_robot}_{source_task.replace('-', '_')}",
    tanh=True,
)

# Load checkpoint if it exists
import os
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from: {checkpoint_path}")
    torch_checkpoint = torch.load(
        f"{checkpoint_path}", map_location=device, weights_only=False
    )
    obs_normalizer == nn.Identity()
    xanchor_normalizer == nn.Identity()
    actor.load_state_dict(torch_checkpoint["actor_state_dict"])
    print("Checkpoint loaded successfully!")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")
    print("Using randomly initialized weights for demonstration.")
    print("To get meaningful results, train a model on h1-stand-v0 first.")

normalize_obs = obs_normalizer.forward
normalize_xanchor = xanchor_normalizer.forward

print(f"Actor parameters: {sum(p.numel() for p in actor.parameters())}")

## Evaluation Functions

In [ ]:
def evaluate():
    """Evaluate the H1 model on the G1 environment."""
    obs_normalizer.eval()
    xanchor_normalizer.eval()
    num_eval_envs = eval_envs.num_envs
    episode_returns = torch.zeros(num_eval_envs, device=device)
    episode_lengths = torch.zeros(num_eval_envs, device=device)
    done_masks = torch.zeros(num_eval_envs, dtype=torch.bool, device=device)
    
    obs, xanchor = eval_envs.reset()
    for _ in range(eval_envs.max_episode_steps):
        with torch.no_grad(), autocast(
            device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
        ):  
            actions = actor(obs, xanchor)

        next_obs, rewards, dones, _ , next_xanchor = eval_envs.step(actions.float())
        episode_returns = torch.where(
            ~done_masks, episode_returns + rewards, episode_returns
        )
        episode_lengths = torch.where(~done_masks, episode_lengths + 1, episode_lengths)
        done_masks = torch.logical_or(done_masks, dones)
        if done_masks.all():
            break
        obs = next_obs
        xanchor = next_xanchor

    obs_normalizer.train()
    xanchor_normalizer.train()
    return episode_returns.mean().item(), episode_lengths.mean().item()

In [ ]:
import tempfile
import imageio
import base64
from IPython.display import display, HTML


def frames_to_video_html(frames, fps=30):
    """
    Convert a list of numpy arrays to an HTML5 video element.

    Args:
        frames (list): List of numpy arrays representing video frames
        fps (int): Frames per second for the video

    Returns:
        HTML object containing the video element
    """
    # Create a temporary file to store the video
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as temp_file:
        temp_filename = temp_file.name

    # Save frames as video
    imageio.mimsave(temp_filename, frames, fps=fps)

    # Read the video file and encode it to base64
    with open(temp_filename, "rb") as f:
        video_data = f.read()
    video_b64 = base64.b64encode(video_data).decode("utf-8")

    # Create HTML video element
    video_html = f"""
    <video width="640" height="480" controls>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    """

    # Clean up the temporary file
    os.unlink(temp_filename)

    return HTML(video_html)


def render_with_rollout():
    """Render a rollout of the H1 model on the G1 environment."""
    obs_normalizer.eval()
    xanchor_normalizer.eval()

    # Quick rollout for rendering
    if env_type == "humanoid_bench":
        obs, xanchor = render_env.reset()
        renders = [render_env.render()]

    for i in range(render_env.max_episode_steps):
        with torch.no_grad(), autocast(
            device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
        ):
            obs = normalize_obs(obs)
            xanchor = normalize_xanchor(xanchor)
            actions = actor(obs, xanchor)
        next_obs, _, done, _, next_xanchor = render_env.step(actions.float())
        if i % 2 == 0:
            if env_type == "humanoid_bench":
                renders.append(render_env.render())
        if done.any():
            break
        obs = next_obs
        xanchor = next_xanchor

    obs_normalizer.train()
    xanchor_normalizer.train()
    video_html = frames_to_video_html(renders, fps=30)
    display(video_html)


## Run Evaluation

Evaluate the H1-trained model on the G1 environment.

In [ ]:
mean_return, mean_length = evaluate()
print(f"\n=== Multi-Embodiment Evaluation Results ===")
print(f"Source: {source_robot}-{source_task}")
print(f"Target: {target_robot}-{target_task}")
print(f"Mean Episode Return: {mean_return:.2f}")
print(f"Mean Episode Length: {mean_length:.2f}")
print("=========================================\n")

## Visualize Performance

Render a video showing how the H1 model performs on the G1 robot.

In [ ]:
render_with_rollout()

## Analysis Notes

### ⚠️ Current Limitation: Dimension Mismatch

This notebook **cannot currently perform direct H1→G1 transfer** due to architectural constraints:

**The Problem:**
- **H1 EGNN** outputs:  - one action per H1 joint
- **G1 environment** expects:  - one action per G1 joint
- The output dimension is hardcoded in  based on 

**Why EGNN has this limitation:**
- The EGNN graph structure is built specifically for one robot type
- Node embeddings are learned per-joint, tied to the robot's topology
- The final output layer maps from hidden_dim → 1 action per node (joint)

### Potential Solutions

To enable true cross-embodiment transfer, you would need:

1. **Joint-Name Mapping Approach:**
   - Map H1's 19 joints to corresponding G1 joints by name (hip, knee, etc.)
   - Zero-pad or freeze the 18 extra G1 hand joints
   - Requires adding an adapter layer after the H1 EGNN

2. **Shared Multi-Robot Training:**
   - Train a single model on both H1 and G1 simultaneously
   - Use a flexible architecture that handles variable joint counts
   - Share weights for common joint types (hip, knee, elbow, etc.)

3. **Universal Robot Embedding:**
   - Create a unified action space covering all possible joints
   - Use masking to ignore joints not present in each robot
   - More complex but enables true zero-shot transfer

### What This Notebook Shows

This notebook serves as:
- A template for **single-robot evaluation**
- Documentation of the **architectural constraints**
- A starting point for implementing one of the solutions above

To use it as intended, train and evaluate on the **same robot** (e.g., both H1 or both G1).
For actual cross-embodiment transfer, architectural modifications are required.